In [ ]:
import os
import time
import pandas as pd
import pickle
from tqdm import tqdm
from nba_api.stats.static import players
from nba_api.stats.endpoints import playercareerstats
import random

In [ ]:
# Function to handle API calls with retries
def safe_api_call(api_function, *args, **kwargs):
    """Retries API calls with exponential backoff if rate limited."""
    retries = 5
    delay = 1  # Start with 1-second delay

    for attempt in range(retries):
        try:
            return api_function(*args, **kwargs)  # Call API function
        except Exception as e:
            print(f"API Error: {e}. Retrying in {delay:.1f}s...")
            time.sleep(delay)
            delay *= 2 + random.uniform(0, 1)  # Exponential backoff with randomness
    print("Max retries reached. Skipping request.")
    return None

In [ ]:
# File paths
csv_file = "nba_career_stats.csv"
cache_file = "nba_api_cache.pkl"

# Load existing data if available
if os.path.exists(csv_file):
    print("Loading existing data from CSV...")
    career_stats_df = pd.read_csv(csv_file) 
    processed_players = set(career_stats_df["Player"].unique())  # Track players already fetched
else:
    print("No existing CSV found. Starting fresh...")
    career_stats_df = pd.DataFrame()
    processed_players = set()
# Load API cache if available
if os.path.exists(cache_file):
    with open(cache_file, "rb") as f:
        api_cache = pickle.load(f)
else:
    api_cache = {}

In [ ]:
# Get all NBA players
nba_players = players.get_players()

# Process players in small batches
batch_size = 50

for i in range(0, len(nba_players), batch_size):
    batch = nba_players[i : i + batch_size]

    for player in tqdm(batch, desc="Fetching Player Stats"):
        player_name = player["full_name"]
        player_id = player["id"]

        # Skip players already processed
        if player_name in processed_players:
            continue

        # Check cache first
        if player_id in api_cache:
            career = api_cache[player_id]
        else:
            career = safe_api_call(playercareerstats.PlayerCareerStats, player_id=player_id)
            api_cache[player_id] = career
            with open(cache_file, "wb") as f:
                pickle.dump(api_cache, f)  # Save updated cache

        if career:
            df = career.get_data_frames()[0]
            df["Player"] = player_name

            # Assign season numbers
            df["YearsExperience"] = range(len(df))  # Season 0, 1, 2, etc.

            # Use pd.concat() to append new data
            career_stats_df = pd.concat([career_stats_df, df], ignore_index=True)

        # Avoid rate limiting
        time.sleep(1.5)  # Increased delay to prevent blocking

    # Save progress after each batch
    career_stats_df.to_csv(csv_file, index=False)
    print("✅ Saved progress to CSV.")

# Final save
career_stats_df.to_csv(csv_file, index=False)
print("✅ Final save to CSV.")

# Display the first few rows
print(career_stats_df.head())